# Cantonese dependency parser — LoRA rank sweep (DEV only)

This self-contained Colab compares LoRA ranks **4, 8, 16, and 32**. It is designed for **Runtime → Run all** and requires no uploaded files. Every rank starts independently from the same Stanza 1.14 Mandarin ELECTRA-large dependency checkpoint. The Transformer base weights remain frozen; LoRA parameters and all parsing layers are trained.

To isolate rank capacity, `alpha/r` is held constant at 2 (`r=4/8/16/32`, `alpha=8/16/32/64`). All other inputs and hyperparameters match the original rank-8 experiment: UD Cantonese-HK r2.18, the fixed custom 803/101 train/dev split, seed 42, frozen Mandarin predicted POS/lemma, learning rates, evaluation interval, and early stopping.

**This notebook never reads or evaluates test.** Rank is selected using DEV CoNLL-2018 LAS only; ties go to the smaller rank. Because this is a one-seed diagnostic, small differences should not be interpreted as stable rank effects without a later multi-seed confirmation.

Use an A100 GPU if possible. Four sequential ELECTRA-large runs can take several hours, although only one model is kept in GPU memory at a time.


In [ ]:
# Pinned Python packages.  Restart the runtime after this cell if Colab asks.
%pip install -q stanza==1.14.0 transformers==4.56.2 peft==0.17.1 huggingface-hub==0.34.4


In [ ]:
from pathlib import Path
import os, json, hashlib, random, shutil, gc, copy, importlib.util, logging
import numpy as np
import torch

SEED = 42
WORK = Path('/content/yue_lora_rank_sweep')
DATA = WORK / 'data'
MODELS = WORK / 'stanza_resources_1.14.0'
OUT = WORK / 'outputs'
HF_HOME = WORK / 'hf_home'
for p in (DATA, MODELS, OUT, HF_HOME): p.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)

RAW_URL = 'https://raw.githubusercontent.com/UniversalDependencies/UD_Cantonese-HK/r2.18/yue_hk-ud-test.conllu'
RAW_SHA256 = 'cbd843a195d0db4cdafbf6fcafb7b7b559afea750411006f4728311e70cc4e2a'
SPLIT_POSITIONS = {'train': [1, 2, 3, 5, 6, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 27, 29, 30, 32, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 48, 49, 51, 52, 53, 54, 55, 56, 57, 58, 60, 61, 62, 63, 64, 65, 66, 68, 70, 71, 72, 74, 76, 77, 78, 79, 80, 81, 84, 85, 86, 87, 88, 89, 90, 92, 93, 94, 98, 99, 100, 102, 103, 104, 107, 108, 109, 110, 111, 112, 113, 115, 116, 118, 119, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 150, 151, 152, 154, 155, 156, 157, 158, 159, 160, 161, 162, 164, 165, 167, 169, 171, 172, 175, 176, 177, 178, 179, 180, 181, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 205, 206, 207, 208, 209, 210, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 223, 226, 227, 229, 230, 234, 235, 237, 238, 239, 240, 243, 244, 246, 248, 249, 251, 252, 253, 254, 255, 256, 257, 258, 259, 261, 263, 264, 265, 266, 267, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 284, 287, 289, 290, 291, 292, 295, 296, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 310, 311, 312, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 339, 340, 341, 342, 343, 344, 345, 346, 347, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 362, 363, 364, 365, 366, 367, 368, 370, 371, 372, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 386, 387, 388, 390, 391, 393, 394, 396, 397, 398, 399, 400, 402, 403, 404, 405, 406, 407, 408, 413, 414, 415, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 428, 429, 431, 433, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 454, 455, 456, 458, 459, 460, 461, 462, 464, 465, 466, 467, 468, 469, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 485, 486, 487, 488, 490, 491, 492, 493, 496, 497, 499, 500, 501, 502, 503, 505, 506, 507, 508, 509, 510, 511, 512, 513, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 524, 525, 526, 527, 528, 529, 531, 532, 533, 534, 535, 537, 538, 539, 540, 541, 543, 544, 545, 547, 548, 549, 550, 551, 552, 553, 554, 555, 556, 557, 558, 559, 560, 561, 562, 563, 564, 565, 566, 568, 569, 570, 571, 572, 573, 575, 576, 578, 580, 581, 582, 583, 584, 585, 587, 588, 589, 590, 591, 592, 593, 596, 597, 598, 600, 601, 602, 603, 605, 606, 607, 608, 609, 611, 612, 613, 614, 615, 616, 617, 618, 619, 620, 622, 623, 624, 625, 626, 629, 630, 631, 632, 635, 636, 637, 638, 639, 641, 642, 643, 644, 645, 647, 649, 650, 651, 652, 654, 655, 656, 657, 658, 659, 660, 661, 662, 664, 665, 666, 667, 668, 669, 670, 671, 674, 675, 676, 677, 678, 679, 681, 682, 683, 687, 689, 690, 691, 692, 694, 696, 697, 698, 699, 700, 702, 703, 704, 705, 706, 708, 709, 710, 711, 712, 713, 714, 715, 717, 718, 719, 720, 721, 723, 724, 725, 727, 730, 732, 733, 735, 736, 737, 738, 739, 740, 741, 742, 743, 745, 746, 747, 749, 750, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 764, 767, 770, 771, 772, 773, 774, 775, 778, 779, 780, 781, 782, 783, 785, 786, 787, 790, 791, 793, 795, 796, 797, 799, 800, 801, 802, 804, 805, 806, 808, 809, 810, 812, 813, 814, 815, 818, 819, 820, 822, 823, 825, 826, 827, 828, 829, 830, 831, 832, 833, 834, 836, 838, 839, 840, 841, 842, 843, 844, 845, 846, 847, 848, 849, 850, 851, 852, 853, 857, 859, 860, 861, 862, 863, 864, 865, 866, 867, 868, 869, 870, 872, 873, 874, 875, 876, 877, 878, 880, 881, 882, 883, 884, 886, 888, 889, 890, 891, 892, 893, 894, 895, 896, 897, 899, 900, 901, 902, 903, 904, 907, 908, 909, 910, 911, 912, 915, 916, 917, 919, 920, 922, 923, 924, 925, 926, 927, 928, 929, 930, 933, 934, 935, 937, 938, 939, 940, 942, 943, 945, 946, 947, 948, 949, 950, 951, 953, 954, 955, 956, 957, 958, 962, 964, 965, 966, 967, 968, 970, 971, 972, 973, 974, 976, 977, 978, 980, 981, 982, 984, 985, 986, 989, 990, 992, 993, 994, 996, 998, 1000, 1001, 1002, 1004], 'dev': [4, 12, 50, 59, 67, 69, 75, 95, 114, 120, 148, 153, 163, 168, 174, 182, 222, 224, 225, 232, 242, 262, 268, 281, 283, 285, 286, 288, 337, 338, 348, 410, 412, 416, 427, 430, 432, 434, 453, 463, 470, 494, 498, 504, 530, 536, 546, 567, 574, 579, 595, 599, 610, 627, 628, 640, 653, 672, 685, 686, 688, 693, 701, 716, 726, 728, 729, 731, 734, 765, 766, 768, 769, 776, 792, 794, 798, 803, 816, 817, 824, 835, 837, 854, 871, 887, 905, 914, 931, 932, 936, 941, 960, 961, 963, 975, 979, 987, 997, 999, 1003], 'test': [7, 26, 28, 31, 33, 45, 47, 73, 82, 83, 91, 96, 97, 101, 105, 106, 117, 132, 149, 166, 170, 173, 204, 211, 228, 231, 233, 236, 241, 245, 247, 250, 260, 282, 293, 294, 297, 309, 313, 361, 369, 373, 385, 389, 392, 395, 401, 409, 411, 452, 457, 484, 489, 495, 542, 577, 586, 594, 604, 621, 633, 634, 646, 648, 663, 673, 680, 684, 695, 707, 722, 744, 748, 751, 763, 777, 784, 788, 789, 807, 811, 821, 855, 856, 858, 879, 885, 898, 906, 913, 918, 921, 944, 952, 959, 969, 983, 988, 991, 995]}
SPLIT_SHA256 = {'train': 'b2d6b96af234f22825bb007a9a2a57ef49619c59747dc582dca7fdb1a4331a2d', 'dev': '41bc28d903457e70a4747307e56b3eb7820f2e6d3ede10ab549fd4a85aef63b7', 'test': '1d7b19ac4c0a75d58a80482413ff9cac63ed18917262c18fa1f92fc0f871c0a0'}
EVAL_URL = 'https://universaldependencies.org/conll18/conll18_ud_eval.py'
EVAL_SHA256 = '1072e02af00b1a56205b5e8216d51dee9b8944a104d80744afaccc78859fcb16'
HF_REPO = 'hfl/chinese-electra-180g-large-discriminator'
# If the earlier 41.32% run recorded a HF commit, paste it here.  None resolves HEAD once,
# records the exact commit, and then forces offline reuse for the rest of this run.
HF_ELECTRA_REVISION = 'd017e219578df8e4885484edbc8969dbdea9cbe0'

RANKS = [4, 8, 16, 32]
ALPHA_TO_RANK = 2                 # alpha/r stays fixed at 2
CFG = dict(dropout=0.10,
           targets=['query', 'value', 'output.dense', 'intermediate.dense'],
           parser_lr=1e-3, lora_lr=2e-5, batch_size=900,
           max_steps=4000, eval_interval=100, patience_steps=600,
           max_grad_norm=1.0)
assert len(RANKS)==len(set(RANKS)) and all(isinstance(r,int) and r>0 for r in RANKS)

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1<<20), b''): h.update(b)
    return h.hexdigest()

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'ELECTRA-large training requires a Colab GPU runtime.'
print('device:', DEVICE, 'config:', CFG)


In [ ]:
# Download the pinned UD source and reconstruct the project's exact custom splits.
import urllib.request
raw_path = DATA / 'yue_hk-ud-test.r2.18.conllu'
urllib.request.urlretrieve(RAW_URL, raw_path)
assert sha256(raw_path) == RAW_SHA256

raw = raw_path.read_text(encoding='utf-8')
blocks = raw.strip().split('\n\n')
assert len(blocks) == 1004
for split in ('train','dev'):
    positions = SPLIT_POSITIONS[split]
    # CoNLL-U requires a blank line between sentence blocks.  The project files
    # also end in one blank line, hence the final two newline characters.
    text = '\n\n'.join(blocks[i-1] for i in positions) + '\n\n'
    path = DATA / f'{split}.conllu'
    path.write_text(text, encoding='utf-8')
    assert sha256(path) == SPLIT_SHA256[split], (split, sha256(path))
    print(split, len(positions), sha256(path))

eval_path = WORK / 'conll18_ud_eval.py'
urllib.request.urlretrieve(EVAL_URL, eval_path)
assert sha256(eval_path) == EVAL_SHA256
spec=importlib.util.spec_from_file_location('official_conll18', eval_path)
official=importlib.util.module_from_spec(spec); spec.loader.exec_module(official)

def conll18(path_gold, path_system):
    return official.evaluate(official.load_conllu_file(str(path_gold)),
                             official.load_conllu_file(str(path_system)))


In [ ]:
# Resolve one exact HF commit and download it into an isolated cache.
from huggingface_hub import snapshot_download
snapshot = Path(snapshot_download(HF_REPO, revision=HF_ELECTRA_REVISION, cache_dir=HF_HOME/'hub'))
HF_COMMIT = snapshot.name
assert len(HF_COMMIT) == 40
print('frozen HF revision:', HF_COMMIT)


In [ ]:
# Download the exact Stanza 1.14 Mandarin processors and verify registry/artifact hashes.
import stanza
from stanza.resources.common import download_resources_json, load_resources_json
from stanza.pipeline.core import DownloadMethod
assert stanza.__version__ == '1.14.0', f'Expected Stanza 1.14.0, found {stanza.__version__}. Restart runtime and rerun.'
download_resources_json(model_dir=str(MODELS))
resources_path=MODELS/'resources.json'
assert sha256(resources_path) == '4e41c1df152146fa26ed0c006a08feea7a60bb3414bb6d57dbda24ad2e3cb99c'
packages={'tokenize':'gsdsimp','pos':'gsdsimp_electra-large','lemma':'gsdsimp_charlm','depparse':'gsdsimp_electra-large'}
stanza.download('zh-hans', model_dir=str(MODELS), package=None, processors=packages, verbose=True)
resources=load_resources_json(model_dir=str(MODELS))
expected_md5={'tokenize':'48f993223d568afedc2893f7cd76719c','pos':'73859e5ec15bedc545d6deafd6ddba94','lemma':'b49edd41abb063a87b125ec53aa5b96c','depparse':'c7ea98d93459b22720337a9dcba1a848'}
artifacts={}
for proc,pkg in packages.items():
    p=MODELS/'zh-hans'/proc/f'{pkg}.pt'
    registry=resources['zh-hans'][proc][pkg]['md5']
    assert registry == expected_md5[proc] and p.exists()
    artifacts[proc]={'path':str(p),'registry_md5':registry,'sha256':sha256(p)}
print(json.dumps(artifacts, indent=2))
# Processor artifacts and the exact Transformer snapshot are now present.  Forbid
# later Hub lookups so every POS/parser load reuses this one cached revision.
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'


In [ ]:
# Run frozen POS/lemma exactly once and build pretagged train/dev caches.
# Gold sentence boundaries and integer-ID FORM tokenization are fed to the pipeline.
# Only LEMMA/UPOS/XPOS/FEATS are replaced; gold HEAD/DEPREL remain training targets.
tagger = stanza.Pipeline(lang='zh-hans', dir=str(MODELS),
    processors={k:v for k,v in packages.items() if k != 'depparse'},
    tokenize_pretokenized=True, use_gpu=True,
    download_method=DownloadMethod.REUSE_RESOURCES, verbose=False)
for proc in ('pos','lemma'):
    model=tagger.processors[proc]._trainer.model
    model.eval()
    for p in model.parameters(): p.requires_grad=False
    assert not any(p.requires_grad for p in model.parameters()), f'{proc} unexpectedly trainable'

def split_blocks(text): return text.strip().split('\n\n')
def integer_rows(block):
    return [line.split('\t') for line in block.splitlines()
            if line and not line.startswith('#') and line.split('\t',1)[0].isdigit()]

def make_pretagged(src, dst, chunk=32):
    bs=split_blocks(src.read_text(encoding='utf-8')); out=[]
    for start in range(0,len(bs),chunk):
        sub=bs[start:start+chunk]
        forms=[[r[1] for r in integer_rows(b)] for b in sub]
        doc=tagger(forms)
        assert len(doc.sentences)==len(sub)
        for block,sent,gold_forms in zip(sub,doc.sentences,forms):
            assert [w.text for w in sent.words] == gold_forms
            pred=iter(sent.words); lines=[]
            for line in block.splitlines():
                if line and not line.startswith('#') and line.split('\t',1)[0].isdigit():
                    c=line.split('\t'); w=next(pred)
                    c[2]=w.lemma or '_'; c[3]=w.upos or '_'; c[4]=w.xpos or '_'; c[5]=w.feats or '_'
                    line='\t'.join(c)
                lines.append(line)
            try: next(pred); raise AssertionError('extra predicted word')
            except StopIteration: pass
            out.append('\n'.join(lines))
    dst.write_text('\n\n'.join(out)+'\n',encoding='utf-8')
    assert len(split_blocks(dst.read_text()))==len(bs)

cache_hashes={}
for split in ('train','dev'):
    dst=DATA/f'{split}.predposlemma.conllu'
    make_pretagged(DATA/f'{split}.conllu',dst)
    cache_hashes[split]=sha256(dst)
    print(split,'frozen cache',cache_hashes[split])
del tagger; gc.collect(); torch.cuda.empty_cache()

EXPECTED_CACHE_SHA256={
 'train':'806b3d42c4f6838d6ba9c6565f29062a0d7b6e136e12d158e4449c3e7e970e89',
 'dev':'e88aa03ff7453469deda68ed30c29dd16769e9839d2010b36abf91faace3bd97'}
assert cache_hashes==EXPECTED_CACHE_SHA256,(cache_hashes,EXPECTED_CACHE_SHA256)
print('Frozen POS/lemma caches match the original r=8 run: OK')


In [ ]:
# Prepare the common Mandarin initialization once.
# Only new Cantonese DEPREL output units are initialized; all original parser
# tensors and old relation slices are kept exactly unchanged.
from stanza.models.pos.vocab import MultiVocab
from stanza.models.common.vocab import VOCAB_PREFIX_SIZE

base_path=Path(artifacts['depparse']['path'])
base_ckpt=torch.load(base_path,map_location='cpu',weights_only=True)
assert base_ckpt.get('model_type','graph')=='graph'
old_vocab=MultiVocab.load_state_dict(base_ckpt['vocab'])
old_units=list(old_vocab['deprel']._id2unit)
train_labels=sorted({row[7] for block in split_blocks((DATA/'train.conllu').read_text())
                     for row in integer_rows(block)})
new_labels=[label for label in train_labels if label not in old_vocab['deprel']]

expanded=copy.deepcopy(base_ckpt)
dep_state=expanded['vocab']['deprel']
dep_state['_id2unit']=old_units+new_labels
dep_state['_unit2id']={unit:i for i,unit in enumerate(dep_state['_id2unit'])}
old_n=len(old_units)-VOCAB_PREFIX_SIZE
new_n=old_n+len(new_labels)
expanded_names=[]
if new_labels:
    for name,tensor in list(expanded['model'].items()):
        if name=='deprel.scorer.W_bilin.weight':
            value=tensor.new_zeros(tensor.shape[0],tensor.shape[1],new_n)
            value[:,:,:old_n]=tensor; expanded['model'][name]=value; expanded_names.append(name)
        elif name=='deprel.scorer.W_bilin.bias':
            value=tensor.new_zeros(new_n)
            value[:old_n]=tensor; expanded['model'][name]=value; expanded_names.append(name)
        elif name.startswith('deprel_linear.') and tensor.ndim in (1,2) and tensor.shape[0]==old_n:
            value=tensor.new_zeros((new_n,)+tuple(tensor.shape[1:]))
            value[:old_n]=tensor; expanded['model'][name]=value; expanded_names.append(name)
    assert expanded_names

def one_dependency_path(model_type):
    dependencies=resources['zh-hans']['depparse']['gsdsimp_electra-large'].get('dependencies',[])
    names=[item['package'] for item in dependencies if item.get('model')==model_type]
    paths=[MODELS/'zh-hans'/model_type/f'{name}.pt' for name in names]
    paths=[path for path in paths if path.exists()]
    if len(paths)!=1: paths=list((MODELS/'zh-hans'/model_type).glob('*.pt'))
    assert len(paths)==1,f'Cannot resolve {model_type}: {paths}'
    return paths[0]

common_args=copy.deepcopy(expanded['config'])
if common_args.get('charlm'):
    common_args['charlm_forward_file']=str(one_dependency_path('forward_charlm').resolve())
    common_args['charlm_backward_file']=str(one_dependency_path('backward_charlm').resolve())
expanded['config']=common_args
expanded['global_step']=0; expanded['last_best_step']=0; expanded['dev_score_history']=[]
expanded_base_path=OUT/'mandarin_electra_expanded_deprel_base.pt'
torch.save(expanded,expanded_base_path,_use_new_zipfile_serialization=False)
del expanded
gc.collect(); torch.cuda.empty_cache()
print('common base prepared:',sha256(expanded_base_path))
print('new DEPREL labels:',new_labels)


In [ ]:
# Train each rank independently from the same Mandarin initialization.
from peft import get_peft_model_state_dict
from stanza.models.common.bert_embedding import load_bert
from stanza.models.common.peft_config import build_peft_wrapper
from stanza.models.common.pretrain import Pretrain
from stanza.models.common.doc import HEAD,DEPREL
from stanza.models.depparse.data import DataLoader,InfiniteBatch
from stanza.models.depparse.trainer import GraphTrainer
from stanza.models.depparse.transition.model import SubtreeCombination
from stanza.models.depparse.utils import predict_dataset
from stanza.utils.conll import CoNLL
import pandas as pd
import matplotlib.pyplot as plt

pretrain_obj=Pretrain(filename=str(one_dependency_path('pretrain'))) if base_ckpt['config'].get('pretrain') else None

def reset_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def predict_to_file(trainer,loader,path):
    trainer.model.eval()
    with torch.inference_mode(): predictions=predict_dataset(trainer,loader)
    loader.doc.set([HEAD,DEPREL],[item for sentence in predictions for item in sentence])
    path.write_text(f'{loader.doc:C}\n\n',encoding='utf-8')
    return path

rank_results=[]
all_histories={}
best_paths={}
for rank in RANKS:
    print(f'\n========== START RANK {rank} (alpha={ALPHA_TO_RANK*rank}) ==========')
    reset_seed()
    rank_dir=OUT/f'rank_{rank}'; rank_dir.mkdir(exist_ok=True)
    initial=torch.load(expanded_base_path,map_location='cpu',weights_only=True)
    args=copy.deepcopy(initial['config'])
    args.update(use_peft=True,bert_finetune=True,lora_rank=rank,lora_alpha=ALPHA_TO_RANK*rank,
        lora_dropout=CFG['dropout'],lora_target_modules=CFG['targets'],lora_modules_to_save=[],
        optim='adamw',second_optim=None,lr=CFG['parser_lr'],
        bert_learning_rate=CFG['lora_lr']/CFG['parser_lr'],bert_start_finetuning=0,
        bert_warmup_steps=0,weight_decay=0.0,bert_weight_decay=0.0,
        max_grad_norm=CFG['max_grad_norm'],batch_size=CFG['batch_size'],seed=SEED,
        enable_gradient_checkpointing=True,augment_nopunct=0.0)

    # Seed a rank-specific adapter for Stanza 1.14 PEFT continuation loading.
    bert,_=load_bert(HF_REPO,enable_gradient_checkpointing=True)
    pefted=build_peft_wrapper(bert,args,logging.getLogger('stanza'),adapter_name='depparse')
    initial['bert_lora']=get_peft_model_state_dict(pefted,adapter_name='depparse')
    initial['config']=args
    compat_path=rank_dir/f'mandarin_electra_lora_r{rank}_init.pt'
    torch.save(initial,compat_path,_use_new_zipfile_serialization=False)
    del bert,pefted,initial
    gc.collect(); torch.cuda.empty_cache()

    load_args=dict(args)
    load_args.pop('transition_subtree_combination',None)
    load_args['charlm_forward_file']=str(one_dependency_path('forward_charlm').resolve())
    load_args['charlm_backward_file']=str(one_dependency_path('backward_charlm').resolve())
    trainer=GraphTrainer.load(str(compat_path),pretrain=pretrain_obj,args=load_args,
                              device=DEVICE,reset_history=True)
    assert isinstance(trainer.args['transition_subtree_combination'],SubtreeCombination)

    loaded=trainer.model.get_params(skip_modules=True)
    for name,old in base_ckpt['model'].items():
        got=loaded[name].detach().cpu()
        if name=='deprel.scorer.W_bilin.weight': assert torch.equal(got[:,:,:old_n],old)
        elif name=='deprel.scorer.W_bilin.bias' or (name.startswith('deprel_linear.') and old.ndim in (1,2) and old.shape[0]==old_n):
            assert torch.equal(got[:old_n],old)
        else: assert got.shape==old.shape and torch.equal(got,old),name

    lora_parameters=[(name,p) for name,p in trainer.model.named_parameters()
                     if name.startswith('bert_model.') and p.requires_grad]
    frozen_parameters=[(name,p) for name,p in trainer.model.named_parameters()
                       if name.startswith('bert_model.') and not p.requires_grad]
    parser_parameters=[(name,p) for name,p in trainer.model.named_parameters()
                       if not name.startswith('bert_model.') and p.requires_grad]
    assert lora_parameters and frozen_parameters and parser_parameters
    assert all(('lora_' in name or 'modules_to_save' in name) for name,_ in lora_parameters)
    lora_parameter_count=sum(p.numel() for _,p in lora_parameters)
    parser_parameter_count=sum(p.numel() for _,p in parser_parameters)

    # Reset again immediately before constructing loaders so data order is the
    # same across ranks and is not affected by rank-specific adapter creation.
    reset_seed()
    train_doc=CoNLL.conll2doc(input_file=str(DATA/'train.predposlemma.conllu'))
    dev_doc=CoNLL.conll2doc(input_file=str(DATA/'dev.predposlemma.conllu'))
    train_loader=DataLoader(train_doc,CFG['batch_size'],trainer.args,pretrain_obj,
        vocab=trainer.vocab,evaluation=False,bert_tokenizer=trainer.model.bert_tokenizer)
    dev_loader=DataLoader(dev_doc,CFG['batch_size'],trainer.args,pretrain_obj,
        vocab=trainer.vocab,evaluation=True,sort_during_eval=True,
        bert_tokenizer=trainer.model.bert_tokenizer)
    infinite=InfiniteBatch(train_loader)

    best_path=rank_dir/f'best_dev_electra_yue_lora_r{rank}.pt'
    history=[]
    prediction=rank_dir/'dev.step0000.conllu'
    predict_to_file(trainer,dev_loader,prediction)
    best_score=conll18(DATA/'dev.conllu',prediction)['LAS'].f1
    best_step=0
    trainer.save(str(best_path))
    last_loss=None
    for step in range(1,CFG['max_steps']+1):
        loss,_=trainer.update(infinite.next_batch(),eval=False)
        last_loss=float(loss); trainer.global_step=step
        if step%20==0: print(f'rank {rank} step {step} loss {last_loss:.5f}')
        if step%CFG['eval_interval']==0:
            prediction=rank_dir/f'dev.step{step:04d}.conllu'
            predict_to_file(trainer,dev_loader,prediction)
            score=conll18(DATA/'dev.conllu',prediction)['LAS'].f1
            row={'rank':rank,'alpha':ALPHA_TO_RANK*rank,'step':step,
                 'dev_conll18_las':score,'loss':last_loss,
                 'prediction_sha256':sha256(prediction)}
            history.append(row)
            print(f'== rank {rank} step {step}: dev LAS {100*score:.2f}% ==')
            if score>best_score:
                best_score=score; best_step=step; trainer.save(str(best_path))
            if step-best_step>=CFG['patience_steps']:
                print('early stop'); break

    history_path=rank_dir/'dev_history.json'
    history_path.write_text(json.dumps(history,indent=2),encoding='utf-8')
    result={'rank':rank,'alpha':ALPHA_TO_RANK*rank,'alpha_over_rank':ALPHA_TO_RANK,
            'best_step':best_step,'best_dev_conll18_las_percent':100*best_score,
            'lora_trainable_parameters':lora_parameter_count,
            'parser_trainable_parameters':parser_parameter_count,
            'checkpoint_sha256':sha256(best_path),'history_sha256':sha256(history_path)}
    rank_results.append(result); all_histories[str(rank)]=history; best_paths[rank]=best_path
    print(json.dumps(result,indent=2))

    del trainer,train_loader,dev_loader,infinite,train_doc,dev_doc
    gc.collect(); torch.cuda.empty_cache()

# Predeclared selection rule: highest DEV LAS; ties go to smaller rank.
selected=min(rank_results,key=lambda row:(-row['best_dev_conll18_las_percent'],row['rank']))
selected_rank=selected['rank']; selected_path=best_paths[selected_rank]
for row in rank_results:
    row['delta_vs_r8_dev_las_points']=row['best_dev_conll18_las_percent']-next(
        item['best_dev_conll18_las_percent'] for item in rank_results if item['rank']==8)
    row['selected_by_dev']=row['rank']==selected_rank

results_df=pd.DataFrame(rank_results).sort_values('rank')
results_df.to_csv(OUT/'rank_sweep_dev_results.csv',index=False)
(OUT/'rank_sweep_dev_results.json').write_text(json.dumps({
    'selection_split':'dev','test_evaluated':False,'seed':SEED,'ranks':RANKS,
    'alpha_policy':'alpha=2*rank, keeping alpha/r=2',
    'selected_rank':selected_rank,'results':rank_results,
    'reference_original_r8_dev_las_percent':74.82337829158638,
    'interpretation_warning':('One seed measures rank sensitivity for this run only. Small '
                              'differences may be smaller than seed-to-seed variation.')},
    ensure_ascii=False,indent=2),encoding='utf-8')

plt.figure(figsize=(7,4.5))
plt.plot(results_df['rank'],results_df['best_dev_conll18_las_percent'],marker='o')
for _,row in results_df.iterrows():
    plt.annotate(f"{row['best_dev_conll18_las_percent']:.2f}",(row['rank'],row['best_dev_conll18_las_percent']),
                 textcoords='offset points',xytext=(0,7),ha='center')
plt.xscale('log',base=2); plt.xticks(RANKS,RANKS)
plt.xlabel('LoRA rank'); plt.ylabel('Best DEV CoNLL-2018 LAS (%)')
plt.title('Cantonese dependency parser: DEV-only LoRA rank sweep')
plt.grid(alpha=.25); plt.tight_layout(); plt.savefig(OUT/'rank_sweep_dev_las.png',dpi=180)
display(results_df)
print('DEV-selected rank:',selected_rank,'checkpoint:',selected_path)
print('TEST HAS NOT BEEN EVALUATED.')


In [ ]:
# Export the DEV-only comparison and the selected rank checkpoint.
from google.colab import files
report_dir=WORK/'rank_sweep_report'; report_dir.mkdir(exist_ok=True)
for path in (OUT/'rank_sweep_dev_results.csv',OUT/'rank_sweep_dev_results.json',
             OUT/'rank_sweep_dev_las.png'):
    shutil.copy2(path,report_dir/path.name)
for rank in RANKS:
    history_path=OUT/f'rank_{rank}'/'dev_history.json'
    shutil.copy2(history_path,report_dir/f'rank_{rank}_dev_history.json')
run_record={'seed':SEED,'ranks':RANKS,'alpha_to_rank':ALPHA_TO_RANK,'config':CFG,
 'data':{'train_sha256':SPLIT_SHA256['train'],'dev_sha256':SPLIT_SHA256['dev'],
         'train_predposlemma_sha256':cache_hashes['train'],
         'dev_predposlemma_sha256':cache_hashes['dev']},
 'initialization':{'stanza_depparse_sha256':sha256(base_path),'hf_revision':HF_COMMIT},
 'selection':{'split':'dev','metric':'CoNLL-2018 LAS','selected_rank':selected_rank},
 'test_evaluated':False}
(report_dir/'run_record.json').write_text(json.dumps(run_record,ensure_ascii=False,indent=2),encoding='utf-8')
report_zip=shutil.make_archive('/content/yue_lora_rank_sweep_dev_results','zip',root_dir=report_dir)

selected_dir=WORK/'selected_rank_bundle'; selected_dir.mkdir(exist_ok=True)
shutil.copy2(selected_path,selected_dir/selected_path.name)
for path in report_dir.iterdir(): shutil.copy2(path,selected_dir/path.name)
selected_zip=shutil.make_archive('/content/yue_lora_rank_sweep_selected_checkpoint','zip',root_dir=selected_dir)
print('DEV report:',report_zip,sha256(Path(report_zip)))
print('Selected-rank checkpoint archive:',selected_zip,sha256(Path(selected_zip)))
files.download(report_zip)
files.download(selected_zip)
